In [4]:
import pandas as pd
import numpy as np

# Read the raw (dirty) CSV file into a DataFrame
df = pd.read_csv("dirty_cafe_sales.csv")


# Show the first 5 rows so we can see what the data looks like
df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [7]:
# Print the number of rows and columns (basic dataset size check)
print("Shape:", df.shape)

# Show column names, non-null counts, and data types (quick overview)
df.info()


Shape: (10000, 8)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [6]:
# Count how many missing (NaN) values are in each column
missing = df.isna().sum()

# Sort from most missing to least missing so we can spot the biggest problems
missing = missing.sort_values(ascending=False)

# Display the missing-value counts
missing



Location            3265
Payment Method      2579
Item                 333
Price Per Unit       179
Total Spent          173
Transaction Date     159
Quantity             138
Transaction ID         0
dtype: int64

In [8]:
# Show the data type of each column (ex: object, int, float, datetime)
df.dtypes


Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

In [9]:
# These columns should be numbers (not text)
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]

# Loop through each numeric column
for c in numeric_cols:
    
    # Try to convert the column to a number
    # If a value cannot be converted (ex: "ERROR"), it becomes NaN
    converted = pd.to_numeric(df[c], errors="coerce")
    
    # Count how many NEW NaNs were created because of bad (non-numeric) values
    bad_count = converted.isna().sum() - df[c].isna().sum()
    
    # Print how many values are breaking numeric conversion
    print(f"{c}: non-numeric values that break numeric type = {bad_count}")


Quantity: non-numeric values that break numeric type = 341
Price Per Unit: non-numeric values that break numeric type = 354
Total Spent: non-numeric values that break numeric type = 329


In [11]:
# Convert the "Transaction Date" column to real datetime values.
# errors="coerce" means: if a value can't be converted, pandas will turn it into NaT (missing datetime).
date_parsed = pd.to_datetime(df["Transaction Date"], errors="coerce")

# Count how many dates became NaT after parsing (these are invalid/unreadable dates)
invalid_dates = date_parsed.isna().sum()

# Count how many values were already truly missing (NaN) before parsing
missing_dates = df["Transaction Date"].isna().sum()

# Print some quick summary stats
print("Total rows:", len(df))
print("Missing dates (true NaN):", missing_dates)
print("Invalid/unparseable dates after parsing:", invalid_dates)

# Show some examples of the raw "Transaction Date" values that failed parsing
# - date_parsed.isna() finds rows where parsing failed (NaT)
# - dropna() removes actual NaN values so we only see the "bad strings"
# - astype(str) makes sure values display as text
# - head(10) shows only the first 10 examples
bad_examples = df.loc[date_parsed.isna(), "Transaction Date"].dropna().astype(str).head(10)
bad_examples



Total rows: 10000
Missing dates (true NaN): 159
Invalid/unparseable dates after parsing: 460


11       ERROR
29       ERROR
33       ERROR
103      ERROR
115    UNKNOWN
142      ERROR
209      ERROR
224    UNKNOWN
259    UNKNOWN
286      ERROR
Name: Transaction Date, dtype: str

In [14]:
# Make a copy of the dataframe 
tmp = df.copy()

# For each column in the list, try to convert the values to numeric (int/float)
# errors="coerce" means: anything that can’t convert becomes NaN
for c in ["Quantity", "Price Per Unit", "Total Spent"]:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

# Count how many rows have Quantity less than or equal to 0 (0 or negative)
print("Quantity <= 0:", (tmp["Quantity"] <= 0).sum())

# Count how many rows have a negative Price Per Unit
print("Price Per Unit < 0:", (tmp["Price Per Unit"] < 0).sum())

# Count how many rows have a negative Total Spent
print("Total Spent < 0:", (tmp["Total Spent"] < 0).sum())

# Show summary statistics (count, mean, std, min, quartiles, max)
# for the three numeric columns
tmp[["Quantity","Price Per Unit","Total Spent"]].describe()


Quantity <= 0: 0
Price Per Unit < 0: 0
Total Spent < 0: 0


,Quantity,Price Per Unit,Total Spent
count,9521.000000,9467.000000,9498.000000
mean,3.028463,2.949984,8.924352
std,1.419007,1.278450,6.009919
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


In [16]:
# Function to count how many times a specific token (like "ERROR" or "UNKNOWN")
# appears in each column of the dataframe
def count_token(token):
    return (
        df.astype(str)  # convert everything to string so string methods work
          .apply(lambda col: col.str.strip()        # remove extra spaces
                            .str.upper()            # make uppercase for matching
                            .eq(token))             # True where value == token
          .sum()  # count Trues per column
    )

# Count "ERROR" occurrences per column
error_counts = count_token("ERROR")

# Count "UNKNOWN" occurrences per column
unknown_counts = count_token("UNKNOWN")

# Build a summary table:
# - NaN = missing values per column
# - ERROR = "ERROR" tokens per column
# - UNKNOWN = "UNKNOWN" tokens per column
# Then sort to show the “worst” columns first
pd.DataFrame({
    "NaN": df.isna().sum(),
    "ERROR": error_counts,
    "UNKNOWN": unknown_counts
}).sort_values(by=["NaN", "ERROR", "UNKNOWN"], ascending=False)



,NaN,ERROR,UNKNOWN
Location,3265,358,338
Payment Method,2579,306,293
Item,333,292,344
Price Per Unit,179,190,164
Total Spent,173,164,165
Transaction Date,159,142,159
Quantity,138,170,171
Transaction ID,0,0,0


In [17]:
# Make a copy so we do not change the original raw dataframe
clean = df.copy()

# Quick preview to confirm the copy exists
clean.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [18]:
# Replace placeholder values with real missing values (NaN)
clean = clean.replace(["ERROR", "UNKNOWN"], np.nan)

# Check missing values after replacement
clean.isna().sum().sort_values(ascending=False)


Location            3961
Payment Method      3178
Item                 969
Price Per Unit       533
Total Spent          502
Quantity             479
Transaction Date     460
Transaction ID         0
dtype: int64

In [19]:
# Columns that should be treated as text/categorical
text_cols = ["Transaction ID", "Item", "Payment Method", "Location"]

# Strip leading/trailing spaces in text columns
for c in text_cols:
    clean[c] = clean[c].astype("string").str.strip()

# Use Title Case for consistent category formatting (optional but helpful)
for c in ["Item", "Payment Method", "Location"]:
    clean[c] = clean[c].str.title()

# Preview standardized columns
clean[text_cols].head()


,Transaction ID,Item,Payment Method,Location
0,TXN_1961373,Coffee,Credit Card,Takeaway
1,TXN_4977031,Cake,Cash,In-Store
2,TXN_4271903,Cookie,Credit Card,In-Store
3,TXN_7034554,Salad,<NA>,<NA>
4,TXN_3160411,Coffee,Digital Wallet,In-Store


In [20]:
# Convert to lowercase so matching is consistent
clean["Item"] = clean["Item"].str.lower()

# Map common misspellings to the correct category name
item_map = {
    "cofee": "coffee",
    "cofe": "coffee",
    "cofffe": "coffee",
    "coffee": "coffee"
}

# Replace typos using the mapping (values not in the map stay the same)
clean["Item"] = clean["Item"].replace(item_map)

# Convert back to Title Case for nicer display
clean["Item"] = clean["Item"].str.title()

# Show the most common items to confirm standardization worked
clean["Item"].value_counts().head(10)


Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
Name: count, dtype: Int64

In [21]:
# Columns that should be numeric
num_cols = ["Quantity", "Price Per Unit", "Total Spent"]

# Remove common symbols like $ and commas, then convert to numeric
for c in num_cols:
    # Turn into string, remove $ and commas, then convert to numeric
    clean[c] = (
        clean[c].astype(str)
              .str.replace("$", "", regex=False)
              .str.replace(",", "", regex=False)
    )
    clean[c] = pd.to_numeric(clean[c], errors="coerce")

# Confirm numeric dtypes
clean[num_cols].dtypes


Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

In [22]:
# Convert Transaction Date to datetime; invalid values become NaT
clean["Transaction Date"] = pd.to_datetime(clean["Transaction Date"], errors="coerce")

# Preview converted dates
clean["Transaction Date"].head()


0   2023-09-08
1   2023-05-16
2   2023-07-19
3   2023-04-27
4   2023-06-11
Name: Transaction Date, dtype: datetime64[us]

In [23]:
# Drop rows where Transaction ID is missing (cannot identify the transaction)
before = len(clean)
clean = clean.dropna(subset=["Transaction ID"]).copy()
after = len(clean)

print("Rows dropped (missing Transaction ID):", before - after)

# Fill missing categorical fields with "Unknown"
clean["Payment Method"] = clean["Payment Method"].fillna("Unknown")
clean["Location"] = clean["Location"].fillna("Unknown")

# Fill missing Quantity using the median 
clean["Quantity"] = clean["Quantity"].fillna(clean["Quantity"].median())

# Fill missing Price Per Unit using the median 
clean["Price Per Unit"] = clean["Price Per Unit"].fillna(clean["Price Per Unit"].median())

# If Total Spent is missing, compute it from Quantity * Price Per Unit 
clean["Total Spent"] = clean["Total Spent"].fillna(clean["Quantity"] * clean["Price Per Unit"])

# Check remaining missing values
clean.isna().sum().sort_values(ascending=False)


Rows dropped (missing Transaction ID): 0


Item                969
Transaction Date    460
Quantity              0
Transaction ID        0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
dtype: int64

In [24]:
# Count duplicate rows before removing them
dup_rows = clean.duplicated().sum()
print("Duplicate rows found:", dup_rows)

# Remove duplicate rows
clean = clean.drop_duplicates()

# Confirm duplicates are removed
print("Duplicate rows after removal:", clean.duplicated().sum())


Duplicate rows found: 0
Duplicate rows after removal: 0


In [25]:
# Confirm final data types match requirements
clean.dtypes


Transaction ID              string
Item                        string
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              string
Location                    string
Transaction Date    datetime64[us]
dtype: object

In [26]:
# The dataset does not include a Tax column, so assume tax is 0.00 for all rows
clean["Tax"] = 0.0

# Calculate what the total should be using the business equation
clean["calc_total"] = (clean["Quantity"] * clean["Price Per Unit"]) + clean["Tax"]

# Preview the calculation
clean[["Quantity", "Price Per Unit", "Tax", "Total Spent", "calc_total"]].head()


,Quantity,Price Per Unit,Tax,Total Spent,calc_total
0,2.0,2.0,0.0,4.0,4.0
1,4.0,3.0,0.0,12.0,12.0
2,4.0,1.0,0.0,4.0,4.0
3,2.0,5.0,0.0,10.0,10.0
4,2.0,2.0,0.0,4.0,4.0


In [27]:
# Compare the given total vs the calculated total
# atol=0.01 allows small rounding differences (ex: 4.999 vs 5.00)
clean["logic_ok"] = np.isclose(clean["Total Spent"], clean["calc_total"], atol=0.01)

# Count how many rows pass vs fail the equation
clean["logic_ok"].value_counts(dropna=False)


logic_ok
True     9267
False     733
Name: count, dtype: int64

In [28]:
# Select only rows where the equation fails
conflicts = clean.loc[
    ~clean["logic_ok"],
    ["Transaction ID", "Item", "Quantity", "Price Per Unit", "Tax", "Total Spent", "calc_total"]
].copy()

# Add a status column like the example in the PDF
conflicts["Status"] = "Logic Conflict"

# Print how many conflict rows exist
print("Number of logic conflicts:", len(conflicts))

# Show the first 10 conflict examples 
conflicts.head(10)


Number of logic conflicts: 733


,Transaction ID,Item,Quantity,Price Per Unit,Tax,Total Spent,calc_total,Status
20,TXN_3522028,Smoothie,3.0,4.0,0.0,20.0,12.0,Logic Conflict
55,TXN_5522862,Cookie,3.0,1.0,0.0,2.0,3.0,Logic Conflict
57,TXN_2080895,Cake,3.0,3.0,0.0,3.0,9.0,Logic Conflict
66,TXN_8501819,Juice,3.0,3.0,0.0,6.0,9.0,Logic Conflict
68,TXN_8427104,Salad,2.0,3.0,0.0,10.0,6.0,Logic Conflict
85,TXN_8035512,Tea,3.0,3.0,0.0,4.5,9.0,Logic Conflict
147,TXN_9336980,Salad,4.0,3.0,0.0,20.0,12.0,Logic Conflict
151,TXN_4031509,<NA>,4.0,3.0,0.0,16.0,12.0,Logic Conflict
153,TXN_6541415,<NA>,3.0,3.0,0.0,12.0,9.0,Logic Conflict
162,TXN_9238666,Cookie,1.0,3.0,0.0,1.0,3.0,Logic Conflict


In [29]:
# Replace Total Spent only for the rows that fail the equation
clean.loc[~clean["logic_ok"], "Total Spent"] = clean.loc[~clean["logic_ok"], "calc_total"]

# Re-check the equation after repair
clean["logic_ok"] = np.isclose(clean["Total Spent"], clean["calc_total"], atol=0.01)

# Confirm all rows now pass
clean["logic_ok"].value_counts(dropna=False)


logic_ok
True    10000
Name: count, dtype: int64

In [30]:
# Remove helper columns used only for verification
# Keep Tax if you want it in final output; otherwise drop it too
final = clean.drop(columns=["calc_total", "logic_ok"])

# Preview final cleaned dataset
final.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Tax
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,0.0
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16,0.0
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19,0.0
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27,0.0
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11,0.0


In [31]:
final = clean.drop(columns=["calc_total","logic_ok"])
final.to_csv("cleaned_cafe_sales1.2.csv", index=False)

final.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Tax
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,0.0
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16,0.0
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19,0.0
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27,0.0
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11,0.0
